In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from piano_tiles_ai.hand_data import load_split
from piano_tiles_ai.hand_model import train_hand_model

DATA_DIR = Path("../data/hands/piano-midi-de")
train_pieces = load_split(DATA_DIR, "train")
val_pieces = load_split(DATA_DIR, "val")
print(len(train_pieces), "training pieces,", len(val_pieces), "validation pieces")

model, history = train_hand_model(train_pieces, val_pieces, epochs=30)

In [ ]:
epochs = [h["epoch"] for h in history]

fig, ax_loss = plt.subplots(figsize=(10, 4))
ax_loss.plot(epochs, [h["train_loss"] for h in history], color="tab:blue")
ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Training loss", color="tab:blue")

ax_acc = ax_loss.twinx()
ax_acc.plot(epochs, [h["val_accuracy"] for h in history], color="tab:orange")
ax_acc.axhline(0.924, color="gray", linestyle="--")
ax_acc.set_ylabel("Validation accuracy", color="tab:orange")

ax_loss.set_title("Learning curves (dashed line: tuned clustering baseline)")
plt.show()

In [ ]:
Path("../models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "../models/hand_lstm.pt")
print(f"Best validation accuracy: {max(h['val_accuracy'] for h in history):.1%}")